# Quickstart

This example runs on the CPU and reads TcTIM (PDB 1TCD) from the installed
MolSysMT resource package. It does not download a PDB or require a notebook
viewer. Provision a compatible environment and connect ElastNetMT as described
in the [installation status](../about/installation.md).

In [ ]:
from importlib.resources import as_file, files
import numpy as np
import molsysmt as msm
from elastnetmt import AnisotropicNetworkModel, GaussianNetworkModel, pyunitwizard as puw

resource = files("molsysmt").joinpath("data/pdb/1tcd.pdb")
with as_file(resource) as path:
    molecular_system = msm.convert(str(path), to_form="molsysmt.MolSys")

## GNM: raw fluctuations and experimental calibration

The normalized Kirchhoff model predicts a dimensionless fluctuation profile.
Fitting experimental B factors returns a scalar multiplier; calibrated results
are plain magnitudes in square angstroms.

In [ ]:
gnm = GaussianNetworkModel(molecular_system, cutoff="7 angstroms", engine="vectorized")
raw = gnm.get_b_factors()
scale, correlation = gnm.fit_to_experimental_b_factors()
fitted = gnm.get_b_factors()
assert raw.shape == (gnm.n_nodes,)
assert np.all(np.isfinite(fitted))
np.testing.assert_allclose(fitted, raw * scale)
scale_again, correlation_again = gnm.fit_to_experimental_b_factors()
np.testing.assert_allclose([scale_again, correlation_again], [scale, correlation])
print(f"Nodes: {gnm.n_nodes}; B-factor correlation: {correlation:.3f}")

In [ ]:
gnm.show_b_factors()

## ANM: directional modes

Six rigid modes are removed. Mode zero is the lowest non-rigid mode.
Eigenvectors are dimensionless; physical displacement is specified separately.

In [ ]:
anm = AnisotropicNetworkModel(molecular_system, cutoff="12 angstroms", engine="vectorized")
modes = anm.get_modes()
assert modes.shape == (3 * anm.n_nodes - 6, anm.n_nodes, 3)
trajectory = anm.trajectory_along_mode(
    mode=0, selection='atom_name=="CA"', amplitude="2 angstroms",
    oscillation_steps=8, interpolation_engine="vectorized",
)
xyz = puw.get_value(msm.get(trajectory, coordinates=True), to_unit="nanometers")
assert xyz.shape == (8, anm.n_nodes, 3)
assert np.all(np.isfinite(xyz))
print(f"Generated {xyz.shape[0]} frames for {xyz.shape[1]} target atoms.")

Continue with the [GNM tutorial](../user/models/gnm/Test_GNM.ipynb),
[ANM tutorial](../user/models/anm/Test_ANM.ipynb), and
[API reference](../../api/index.md).
Interactive overlays are provided by the separate MolSysViewer add-on;
the numerical examples here remain independent of its frontend.